# 01e — Replogle 2022: CPA Gene Programs

**Dataset:** ReplogleWeissman2022_rpe1 (scPerturb-harmonized)  
**Accession:** GSE181897  
**Phase:** 2  
**Date:** 2026-03-11  
**Objective:** Train scvi.external.CPA model; inspect perturbation embeddings; apply CINEMA-OT and compare program membership with CPA.

## Table of Contents

1. [Setup](#setup)
2. [Load Data from 01b](#load)
3. [Background: CPA Model Architecture](#cpa-background)
   - 3a. Why go beyond scVI? The limitation of an unsupervised latent space
   - 3b. The Compositional Perturbation Autoencoder (CPA)
   - 3c. The three-way decomposition: basal + perturbation + covariate
   - 3d. Perturbation embeddings and what they represent
   - 3e. Training: reconstruction loss + disentanglement adversary
   - 3f. In silico perturbation prediction
4. [CPA: Data Preparation](#cpa-prep) — AnnData setup, filter to single perts
5. [CPA: Model Initialization and Training](#cpa-train)
6. [CPA: Perturbation Embeddings](#cpa-embed) — extract, cluster, compare to DE
7. [CPA: In Silico Perturbation Prediction](#cpa-predict) — predict held-out perturbations
8. [Background: CINEMA-OT](#cinemaot-background) — causal inference via optimal transport
   - 8a. The fundamental challenge: confounding in perturbation data
   - 8b. Optimal transport for matching cells
   - 8c. Causal gene programs from the transport plan
9. [CINEMA-OT: Application](#cinemaot-apply) — causal effects for selected perturbations
10. [Program Comparison: CPA vs. CINEMA-OT](#comparison) — shared and divergent programs
11. [Save Results](#save)
12. [Summary](#summary)

<a id='setup'></a>
## 0. Setup

In [ ]:
import anndata as ad
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import scipy.sparse as sp
from scipy.spatial.distance import pdist, squareform
import boto3
import warnings
warnings.filterwarnings('ignore')

import scvi
import pertpy as pt

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, frameon=False)
np.random.seed(42)
scvi.settings.seed = 42

from pathlib import Path
PROC_DIR    = Path('../../data/processed/scperturb')
MODEL_DIR   = Path('../../data/processed/scperturb/cpa_model_rpe1')
RESULTS     = Path('../../results')
FIG_DIR     = RESULTS / 'figures'
TBL_DIR     = RESULTS / 'tables'
for d in [PROC_DIR, MODEL_DIR, FIG_DIR, TBL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

S3_BUCKET  = 'learn-perturb-seq'

# Input: scVI-processed AnnData from 01b (with X_scVI, X_umap_scvi, leiden_scvi)
SCVI_PATH  = PROC_DIR / 'ReplogleWeissman2022_rpe1_scvi.h5ad'
QC_PATH    = PROC_DIR / 'ReplogleWeissman2022_rpe1_qc.h5ad'

# Output: AnnData with CPA and CINEMA-OT program annotations
OUT_PATH   = PROC_DIR / 'ReplogleWeissman2022_rpe1_cpa.h5ad'

# CPA hyperparameters
N_LATENT_BASAL  = 64     # dimensionality of the basal (cell state) latent space
N_LATENT_DRUG   = 64     # dimensionality of the perturbation embedding space
N_HIDDEN        = 256    # hidden layer width for encoder/decoder
CPA_MAX_EPOCHS  = 400    # max epochs (early stopping typically triggers earlier)
CPA_BATCH_SIZE  = 128    # minibatch size

# CINEMA-OT parameters
CINEMAOT_N_PROGRAMS = 10   # number of causal gene programs to infer

print(f'scvi-tools version: {scvi.__version__}')
print(f'pertpy version:     {pt.__version__}')
print('Setup complete.')

<a id='load'></a>
## 1. Load Data from 01b

We load the AnnData produced by `01b_dimred_visualization.ipynb`, which contains the
scVI latent space (`X_scVI`), scVI-corrected UMAP (`X_umap_scvi`), and Leiden
clusters (`leiden_scvi`).

For CPA, we also need:
- **Raw counts** in `adata.raw` — CPA, like scVI, models raw count data
- **`perturbation` column** in `adata.obs` — the perturbation label for each cell
- **Single-perturbation cells only** (`nperts == 1`) — CPA's additive model is
  defined for single-gene perturbations (combinatorial perturbations require a
  separate interaction model)

In [ ]:
# Load the most complete available file (prefer 01b output with scVI embeddings)
load_path = SCVI_PATH if SCVI_PATH.exists() else QC_PATH
if not load_path.exists():
    print(f'{load_path} not found locally. Checking S3...')
    s3 = boto3.client('s3')
    for candidate in [SCVI_PATH, QC_PATH]:
        try:
            s3.download_file(S3_BUCKET, f'processed/scperturb/{candidate.name}', str(candidate))
            load_path = candidate
            print(f'Downloaded {candidate.name} from S3.')
            break
        except Exception:
            continue
    else:
        raise FileNotFoundError(
            'Processed AnnData not found locally or on S3.\n'
            'Run 01a_qc_preprocessing.ipynb and 01b_dimred_visualization.ipynb first.'
        )

adata = sc.read_h5ad(load_path)
print(f'Loaded: {load_path.name}')
print(f'  Cells: {adata.n_obs:,}  Genes: {adata.n_vars:,}')
print(f'  Raw counts: {adata.raw is not None}')
print(f'  obsm keys:  {list(adata.obsm.keys())}')
print(f'  obs columns: {adata.obs.columns.tolist()}')
print()

# ---- Filter to clean single-perturbation cells ----------------------------
# CPA's additive assumption applies only to singly-perturbed cells.
# Ambiguous guide assignments add noise to the perturbation label.
mask_single = adata.obs['nperts'] == 1
mask_clean  = (~adata.obs['guide_ambiguous']) if 'guide_ambiguous' in adata.obs.columns \
              else pd.Series(True, index=adata.obs_names)

adata = adata[mask_single & mask_clean].copy()
print(f'After filtering to single-pert, clean-guide cells: {adata.n_obs:,}')

# ---- Identify perturbations with enough cells for reliable training --------
# CPA benefits from at least 50 cells per perturbation to learn meaningful
# perturbation embeddings. We subset to perturbations meeting this threshold.
MIN_CELLS_CPA = 50
pert_counts   = adata.obs['perturbation'].value_counts()
perts_keep    = pert_counts[pert_counts >= MIN_CELLS_CPA].index.tolist()

adata = adata[adata.obs['perturbation'].isin(perts_keep)].copy()
print(f'After filtering to perts with >= {MIN_CELLS_CPA} cells: {adata.n_obs:,} cells')
print(f'  Perturbations retained: {adata.obs["perturbation"].nunique():,}')
print(f'  Control cells: {(adata.obs["perturbation"] == "control").sum():,}')

<a id='cpa-background'></a>
## 2. Background: CPA Model Architecture

### 3a. Why go beyond scVI? The limitation of an unsupervised latent space

The scVI model trained in `01b` is a *unsupervised* generative model — it knows nothing
about which cells were perturbed or what guide RNA they received. The scVI latent space
encodes the full transcriptional state of each cell, which includes:

- The **basal cell state** (what the cell "would look like" without perturbation)
- The **perturbation effect** (the additional expression changes caused by the knockdown)
- **Confounders** (cell cycle phase, batch effects, stochastic variation)

These contributions are *entangled* in the scVI embedding. If we cluster cells in scVI
space, we cannot tell whether a cluster reflects a shared perturbation effect, a shared
basal cell state, or a shared confounder.

**The goal of CPA is to disentangle these contributions.** By providing the perturbation
label during training, CPA can learn separate representations for:
1. The basal state (what the cell looks like without perturbation)
2. The perturbation effect (what the knockdown adds on top of the basal state)
3. Technical covariates (batch effects, if any)

This disentanglement enables two uniquely powerful analyses:
- **Perturbation embeddings:** A vector per perturbation (not per cell) in a shared
  embedding space where geometric relationships reflect biological similarity.
- **In silico perturbation prediction:** Given a cell's basal state, we can predict
  what its transcriptome would look like under any perturbation — including combinations
  not present in the experiment.

---

### 3b. The Compositional Perturbation Autoencoder (CPA)

CPA was introduced by Lotfollahi et al. (2023) as an extension of the variational
autoencoder framework. The key references are:

> Lotfollahi, M. et al. (2023). Predicting cellular responses to complex perturbations
> in high-throughput screens. *Molecular Systems Biology* 19, e11517.

> Lotfollahi, M. et al. (2021). Compositional perturbation autoencoder for single-cell
> response modeling. *bioRxiv* 2021.04.14.439903.

The model is implemented in `scvi-tools` as `scvi.external.CPA`.

---

### 3c. The three-way decomposition

CPA represents each cell's gene expression as the **sum of three latent vectors**:

```
  z_observed = z_basal + z_perturbation + z_covariate
```

Each component is a vector in the same latent space (same dimensionality):

**1. Basal state vector `z_basal` (per cell):**
- Encoded from the cell's raw counts by a standard VAE encoder
- Represents "what is this cell's transcriptional state, independent of perturbation?"
- Should capture: cell cycle phase, differentiation state, stochastic variation
- Should NOT capture: perturbation effects, batch effects

**2. Perturbation embedding `z_pert` (per perturbation):**
- A *learnable embedding matrix* with one row per perturbation label
- All cells with the same perturbation label share the same `z_pert` vector
- Represents "what does this perturbation add on top of the basal state?"
- Dimension: `n_perturbations × n_latent_drug`
- This is the key output we will analyse — it gives us one vector per gene knockdown

**3. Covariate embedding `z_cov` (per batch / covariate):**
- Learnable embedding for technical covariates (batch, sex, cell line, etc.)
- For the Replogle RPE1 dataset (single batch), this is unused

**The decoder** then takes the sum `z_basal + z_pert + z_cov` and reconstructs the
expected raw counts via a negative binomial output distribution (same as scVI).

---

### 3d. Perturbation embeddings and what they represent

The perturbation embedding matrix $W \in \mathbb{R}^{n_{\text{pert}} \times n_{\text{latent}}}$
is one of the most valuable outputs of the CPA model.

**Geometric interpretation:**

- Two perturbations with **similar embeddings** (close in Euclidean or cosine distance)
  are predicted to have similar transcriptional effects. Critically, this similarity is
  learned from the data rather than derived from DE profiles — it captures the structure
  of the expression manifold, not just which genes are differentially expressed.

- **Arithmetic in embedding space:** Because the decoder is a nonlinear function of
  the sum of embeddings, perturbation embedding space can in principle support vector
  arithmetic analogous to word2vec:
  - `z_pert(A) + z_pert(B) ≈ z_pert(A+B)` — if A and B have additive effects
  - `z_pert(A) - z_pert(B)` — the "differential effect" of A relative to B

- **Clustering perturbation embeddings** reveals functional gene groups — perturbations
  that cluster together likely operate in the same pathway or regulatory circuit.

- **Comparing CPA clusters to DE-based clusters (from 01c):** Both approaches aim to
  group perturbations by biological similarity, but they use different information:
  - DE-based: similarity of fold-change profiles across individual genes
  - CPA-based: similarity of positions in a learned nonlinear manifold
  CPA may capture latent relationships that are not obvious from pairwise DE comparisons.

---

### 3e. Training: reconstruction loss + disentanglement adversary

CPA uses a two-part training objective:

**1. Reconstruction loss (same as scVI ELBO):**
$$\mathcal{L}_{\text{recon}} = \mathbb{E}_{q(z_b|x)}[\log p(x \mid z_b + z_p + z_c)] - \text{KL}[q(z_b|x) \| p(z_b)]$$

This ensures the model can reconstruct the observed counts from the decomposed representation.

**2. Adversarial disentanglement loss:**

The key challenge is ensuring that `z_basal` does NOT encode perturbation information.
Without explicit enforcement, the encoder might "cheat" by storing perturbation identity
in the basal vector (which is richer and more expressive than the fixed perturbation embedding).

CPA prevents this using an **adversarial classifier**:
- A small discriminator network tries to predict the perturbation label from `z_basal`
- The encoder is trained to *fool* this discriminator (make `z_basal` uninformative about perturbation)
- This is a minimax game: encoder minimises, discriminator maximises classification accuracy

$$\mathcal{L}_{\text{adv}} = - \lambda \cdot \mathbb{E}[\log P(\text{pert} \mid z_b)]$$

The adversary forces the basal state to be perturbation-agnostic — the only way to
reconstruct the perturbed expression is to use the perturbation embedding.

**The balance** between reconstruction and adversarial loss (controlled by $\lambda$)
is a key hyperparameter. Too much adversary → basal state loses biological information;
too little → perturbation effects leak into `z_basal`.

---

### 3f. In silico perturbation prediction

Once trained, the CPA model can predict what a cell would look like under any perturbation
in the embedding space — even combinations not observed during training:

1. Encode a control cell to get its basal state `z_basal`
2. Look up the perturbation embedding `z_pert` for the target perturbation
3. Decode `z_basal + z_pert` to get the predicted perturbed gene expression

This enables:
- **Perturbation interpolation:** Predict the effect of a new combination
  (e.g., A + B) from the individual embeddings of A and B
- **Perturbation extrapolation:** Predict the effect of a perturbation in a cell type
  not present in the training data
- **Comparison to measured data:** Validate predictions against held-out perturbations
  to assess the model's generalisation ability

<a id='cpa-prep'></a>
## 3. CPA: Data Preparation

CPA requires:
1. **Raw counts** — the model has its own internal normalisation
2. **HVG subset** — same 2,000 HVGs used for scVI reduces memory and speeds training
3. **`perturbation` column** — the categorical perturbation label
4. **`control_group` specification** — which label represents unperturbed cells

The `setup_anndata` call registers these fields with the model.

In [ ]:
# ---- Recover raw counts and subset to HVGs ---------------------------------
# Raw counts are stored in adata.raw (saved by 01a before normalisation)
if adata.raw is None:
    raise ValueError(
        'adata.raw is missing. Raw counts are required for CPA.\n'
        'Re-run 01a_qc_preprocessing.ipynb to regenerate the processed file.'
    )

adata_raw = adata.raw.to_adata()

# Restrict to HVGs selected in 01a
hvg_genes = adata.var_names[adata.var['highly_variable']]
hvg_in_raw = hvg_genes[hvg_genes.isin(adata_raw.var_names)]
adata_cpa = adata_raw[:, hvg_in_raw].copy()

# Carry over obs columns needed for CPA (perturbation label + QC flags)
adata_cpa.obs = adata.obs.copy()

# Sanity check: verify integer raw counts
X_sample = adata_cpa.X[:50, :50]
if sp.issparse(X_sample):
    X_sample = X_sample.toarray()
print(f'CPA input AnnData: {adata_cpa.shape}')
print(f'  Value range (sample): [{X_sample.min():.0f}, {X_sample.max():.0f}]')
print(f'  Are values integers?  {np.allclose(X_sample, X_sample.astype(int))}')

# ---- Encode the perturbation column as a categorical -----------------------
# CPA needs the perturbation column to be a categorical dtype so it can
# build the embedding matrix with the right number of rows.
adata_cpa.obs['perturbation'] = pd.Categorical(adata_cpa.obs['perturbation'])
n_perts_unique = adata_cpa.obs['perturbation'].nunique()
print(f'\nUnique perturbation labels (including control): {n_perts_unique}')
print(f'Perturbation embedding matrix will be: {n_perts_unique} × {N_LATENT_DRUG}')

# ---- Register AnnData with CPA via setup_anndata --------------------------
# perturbation_key: the obs column containing perturbation labels
# control_group:    the label used for unperturbed cells
#
# Under the hood, setup_anndata:
#   - Registers the count matrix (X) as the input data
#   - Creates a one-hot or index representation of perturbation labels
#   - Stores size factors for library-size normalisation during decoding
scvi.external.CPA.setup_anndata(
    adata_cpa,
    perturbation_key='perturbation',
    control_group='control',
)
print('\nAnnData registered with CPA.')
print(f'  Registered fields: {list(adata_cpa.uns.get("_scvi", {}).get("data_registry", {}).keys())}')

<a id='cpa-train'></a>
## 4. CPA: Model Initialization and Training

### Hyperparameter choices

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| `n_latent` | 64 | Basal state dimensionality; higher than scVI because we want richer cell-state representation |
| `n_latent_drug` | 64 | Perturbation embedding dimensionality; matches `n_latent` so arithmetic is well-defined |
| `n_hidden` | 256 | Hidden layer width; larger than scVI to capture complex perturbation-state interactions |
| `max_epochs` | 400 | Longer than scVI; adversarial training needs more steps to converge |
| `batch_size` | 128 | Smaller than scVI; each minibatch should contain multiple examples per perturbation |

### Expected training time

On an `r7i.4xlarge` (CPU only): ~60–120 minutes for 200–400 epochs.  
On a `g4dn.2xlarge` (T4 GPU): ~10–20 minutes.  
We save and cache the model to avoid re-training.

In [ ]:
# ---- Initialise the CPA model ---------------------------------------------
# n_latent:      dimensionality of the basal (cell-state) latent space
# n_latent_drug: dimensionality of the perturbation embedding space
# n_hidden:      width of encoder and decoder hidden layers
# n_layers:      depth of encoder and decoder (1 = single hidden layer)
model = scvi.external.CPA(
    adata_cpa,
    n_latent=N_LATENT_BASAL,
    n_latent_drug=N_LATENT_DRUG,
    n_hidden=N_HIDDEN,
)

print('CPA model initialised.')
total_params = sum(p.numel() for p in model.module.parameters())
print(f'  Total parameters: {total_params:,}')
print(f'  Encoder:   {adata_cpa.n_vars} genes → {N_HIDDEN} hidden → {N_LATENT_BASAL} basal latent')
print(f'  Decoder:   {N_LATENT_BASAL} + {N_LATENT_DRUG} latent → {N_HIDDEN} hidden → {adata_cpa.n_vars} genes')
print(f'  Perturbation embedding: {n_perts_unique} × {N_LATENT_DRUG}')
print(f'  Adversarial discriminator: {N_LATENT_BASAL} → {n_perts_unique}')

# ---- Load or train the model -----------------------------------------------
if MODEL_DIR.exists() and (MODEL_DIR / 'model.pt').exists():
    print(f'\nPre-trained CPA model found at {MODEL_DIR}. Loading...')
    model = scvi.external.CPA.load(str(MODEL_DIR), adata=adata_cpa)
    print('Model loaded successfully.')
else:
    print(f'\nTraining CPA (max_epochs={CPA_MAX_EPOCHS}, batch_size={CPA_BATCH_SIZE})...')
    print('Note: training with adversarial disentanglement takes longer than scVI.')
    print('      Monitor the adversary loss — it should initially rise then fall.')
    model.train(
        max_epochs=CPA_MAX_EPOCHS,
        batch_size=CPA_BATCH_SIZE,
        early_stopping=True,
        early_stopping_patience=25,    # more patience than scVI (adversary needs time)
        early_stopping_monitor='reconstruction_loss_validation',
        train_size=0.9,
        check_val_every_n_epoch=5,
    )
    model.save(str(MODEL_DIR), overwrite=True)
    print(f'\nModel saved to {MODEL_DIR}')

# ---- Plot training history ------------------------------------------------
# A well-trained CPA model shows:
#   1. Reconstruction loss decreasing (model learns to explain the data)
#   2. Adversary loss initially high then decreasing (encoder learns to resist)
#   3. No divergence between train and validation reconstruction loss (no overfitting)
try:
    history = model.history
    history_keys = list(history.keys())
    print(f'\nTraining history keys: {history_keys}')

    # Identify reconstruction and adversary loss columns
    recon_key = next((k for k in history_keys if 'reconstruction' in k.lower() and 'train' in k.lower()), None)
    recon_val_key = next((k for k in history_keys if 'reconstruction' in k.lower() and 'val' in k.lower()), None)
    adv_key   = next((k for k in history_keys if 'advers' in k.lower() or 'disent' in k.lower()), None)
    elbo_key  = next((k for k in history_keys if 'elbo' in k.lower() and 'train' in k.lower()), None)

    n_plots = sum([recon_key is not None, adv_key is not None, elbo_key is not None])
    if n_plots == 0:
        print('No recognized loss keys in history — skipping plot.')
    else:
        fig, axes = plt.subplots(1, max(n_plots, 1), figsize=(6 * max(n_plots, 1), 4))
        if n_plots == 1:
            axes = [axes]

        ax_idx = 0
        if recon_key:
            axes[ax_idx].plot(
                history[recon_key].index, history[recon_key].values,
                label='Train reconstruction', color='steelblue'
            )
            if recon_val_key:
                axes[ax_idx].plot(
                    history[recon_val_key].index, history[recon_val_key].values,
                    label='Validation reconstruction', color='tomato'
                )
            axes[ax_idx].set_xlabel('Epoch')
            axes[ax_idx].set_ylabel('Reconstruction loss')
            axes[ax_idx].set_title('CPA reconstruction loss')
            axes[ax_idx].legend(fontsize=8)
            ax_idx += 1

        if adv_key:
            axes[ax_idx].plot(
                history[adv_key].index, history[adv_key].values,
                color='green', label=adv_key
            )
            axes[ax_idx].set_xlabel('Epoch')
            axes[ax_idx].set_ylabel('Adversary loss')
            axes[ax_idx].set_title('CPA adversarial disentanglement')
            axes[ax_idx].legend(fontsize=8)
            ax_idx += 1

        if elbo_key and ax_idx < len(axes):
            axes[ax_idx].plot(
                history[elbo_key].index, history[elbo_key].values,
                color='purple', label=elbo_key
            )
            axes[ax_idx].set_xlabel('Epoch')
            axes[ax_idx].set_ylabel('ELBO')
            axes[ax_idx].set_title('ELBO (combined objective)')
            axes[ax_idx].legend(fontsize=8)

        plt.suptitle('CPA training history — ReplogleWeissman2022_rpe1', y=1.02)
        plt.tight_layout()
        plt.savefig(FIG_DIR / '01e_cpa_training_history.pdf', bbox_inches='tight')
        plt.show()
except Exception as e:
    print(f'Could not plot training history: {e}')

<a id='cpa-embed'></a>
## 5. CPA: Perturbation Embeddings

The perturbation embedding matrix is the key output of CPA for hypothesis generation.
Each row is a `n_latent_drug`-dimensional vector that represents how that perturbation
shifts the cell state in the decoded expression space.

We perform three analyses on the embeddings:

1. **PCA + UMAP of perturbation embeddings** — visualise the landscape of perturbation
   effects; perturbations that cluster together share downstream biology

2. **Cosine distance matrix** — pairwise similarities between perturbation embeddings

3. **Comparison with DE-based similarity (from 01c)** — do the two methods agree on
   which perturbations are most similar? Disagreements are informative: CPA may capture
   latent similarities not visible in the fold-change profile.

In [ ]:
# ---- Extract perturbation embeddings --------------------------------------
# get_drug_embeddings returns a DataFrame: perturbation x n_latent_drug
# The perturbation embeddings are the rows of the learned embedding matrix W.
# 'control' is included with a near-zero vector (the model learns that
# the control perturbation adds no shift to the basal state).

pert_embed_df = model.get_drug_embeddings()
print(f'Perturbation embeddings: {pert_embed_df.shape}')
print(f'  (perturbations × n_latent_drug)')
print(f'\nFirst 5 perturbations in embedding table:')
print(pert_embed_df.head())

# Separate control from the perturbations
ctrl_embed    = pert_embed_df.loc['control'].values if 'control' in pert_embed_df.index else None
pert_only_df  = pert_embed_df.drop('control', errors='ignore')
print(f'\nNon-control perturbation embeddings: {pert_only_df.shape}')

# ---- Sanity check: control embedding should be near zero ------------------
# The control perturbation's embedding represents "no perturbation" —
# the model should have learned this is a near-zero vector.
if ctrl_embed is not None:
    ctrl_norm = np.linalg.norm(ctrl_embed)
    pert_norms = np.linalg.norm(pert_only_df.values, axis=1)
    print(f'\nEmbedding norm summary:')
    print(f'  Control embedding norm: {ctrl_norm:.4f}')
    print(f'  Mean perturbation embedding norm: {pert_norms.mean():.4f}')
    print(f'  Std perturbation embedding norm:  {pert_norms.std():.4f}')
    if ctrl_norm < pert_norms.mean() * 0.3:
        print('  [Good] Control norm is much smaller than perturbation norms.')
    else:
        print('  [Note] Control norm is not near zero — the adversary may not have fully converged.')

In [ ]:
# ---- Build an AnnData of perturbation embeddings for visualisation ---------
# This is conceptually similar to what we did in 01c with the LFC matrix:
# each "cell" is a perturbation, each "gene" is a CPA embedding dimension.
adata_pe = ad.AnnData(
    X=pert_only_df.values.astype(np.float32),
    obs=pd.DataFrame(index=pert_only_df.index),
)

# Add per-perturbation metadata from 01c if available
de_summary_path = TBL_DIR / '01c_de_summary.csv'
if de_summary_path.exists():
    de_meta = pd.read_csv(de_summary_path).set_index('perturbation')
    adata_pe.obs = adata_pe.obs.join(de_meta[['n_sig', 'n_up', 'n_down', 'n_cells']], how='left')
    print('Joined DE summary from 01c.')
else:
    print('01c_de_summary.csv not found — skipping DE metadata join.')

# Add cell-count data from adata.obs
if 'n_cells' not in adata_pe.obs.columns:
    adata_pe.obs['n_cells'] = (
        adata.obs['perturbation'].value_counts()
        .reindex(adata_pe.obs_names)
        .values
    )

# ---- PCA on perturbation embeddings ---------------------------------------
# The embedding dimensions are already in a compact latent space (n_latent_drug),
# so we do a light PCA reduction for UMAP computation.
N_PC_PE = min(30, pert_only_df.shape[0] - 1, pert_only_df.shape[1] - 1)
sc.tl.pca(adata_pe, n_comps=N_PC_PE)
sc.pp.neighbors(adata_pe, n_neighbors=15, use_rep='X_pca')
sc.tl.umap(adata_pe, random_state=42)
sc.tl.leiden(adata_pe, resolution=0.6, key_added='leiden_cpa', random_state=42)

n_cpa_clusters = adata_pe.obs['leiden_cpa'].nunique()
print(f'\nLeiden clustering on CPA embeddings: {n_cpa_clusters} clusters')
print('Perturbations per cluster:')
for cl in sorted(adata_pe.obs['leiden_cpa'].unique(), key=int):
    members = adata_pe.obs[adata_pe.obs['leiden_cpa'] == cl].index.tolist()
    print(f'  Cluster {cl}: {len(members):3d} perturbations')

# ---- Visualise perturbation embedding UMAP --------------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 6.5))

# Left: Leiden clusters on CPA embedding UMAP
sc.pl.umap(adata_pe, color='leiden_cpa', ax=axes[0], show=False,
           title='CPA perturbation embeddings: Leiden clusters',
           legend_loc='right margin', s=20)

# Right: colour by number of significant DE genes (from 01c)
color_key = 'n_sig' if 'n_sig' in adata_pe.obs.columns else 'n_cells'
sc.pl.umap(adata_pe, color=color_key, ax=axes[1], show=False,
           title=f'CPA perturbation embeddings: coloured by {color_key}',
           color_map='viridis', s=20)

plt.suptitle('CPA perturbation embedding space — ReplogleWeissman2022_rpe1', y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / '01e_cpa_embedding_umap.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ---- Compare CPA embedding similarity to DE-based similarity (from 01c) ---
#
# Both 01c (cosine similarity of LFC profiles) and CPA (Euclidean distance in
# embedding space) aim to capture which perturbations have similar effects.
# Comparing them answers: "what does CPA capture that LFC profiles miss?"
#
# Methodology:
#   1. Compute pairwise cosine similarities in CPA embedding space
#   2. Load the DE-based cosine similarity matrix from 01c
#   3. For perturbations present in both matrices, scatter-plot the two similarities
#   4. High agreement → both methods capture the same biology
#      Outliers above diagonal → CPA finds pairs similar that DE does not
#      Outliers below diagonal → DE finds pairs similar that CPA does not

# CPA pairwise cosine similarity
cpa_cosine = 1 - squareform(pdist(pert_only_df.values, metric='cosine'))
cpa_sim_df = pd.DataFrame(cpa_cosine, index=pert_only_df.index, columns=pert_only_df.index)

# Load DE-based similarity from 01c (if available)
de_sim_path = TBL_DIR / '01c_cosine_similarity.csv.gz'
if de_sim_path.exists():
    de_sim_df = pd.read_csv(de_sim_path, index_col=0)
    print(f'Loaded DE-based similarity matrix: {de_sim_df.shape}')

    # Find common perturbations
    common_perts = sorted(set(cpa_sim_df.index) & set(de_sim_df.index))
    print(f'Common perturbations in both matrices: {len(common_perts)}')

    if len(common_perts) >= 10:
        # Extract upper triangle of each similarity matrix (avoid duplicate pairs)
        idx = np.array([(i, j)
                        for i in range(len(common_perts))
                        for j in range(i + 1, len(common_perts))])
        cpa_vals = np.array([
            cpa_sim_df.loc[common_perts[i], common_perts[j]] for i, j in idx
        ])
        de_vals = np.array([
            de_sim_df.loc[common_perts[i], common_perts[j]] for i, j in idx
        ])

        # Subsample for plotting
        n_plot = min(50_000, len(cpa_vals))
        rng = np.random.default_rng(42)
        plot_idx = rng.choice(len(cpa_vals), size=n_plot, replace=False)

        r = np.corrcoef(cpa_vals[plot_idx], de_vals[plot_idx])[0, 1]

        fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

        # Scatter: CPA similarity vs. DE similarity
        axes[0].scatter(
            cpa_vals[plot_idx], de_vals[plot_idx],
            s=2, alpha=0.2, color='steelblue', rasterized=True
        )
        axes[0].plot([-0.5, 1], [-0.5, 1], 'r--', lw=0.8, label='y = x')
        axes[0].set_xlabel('CPA embedding cosine similarity')
        axes[0].set_ylabel('DE profile cosine similarity (01c)')
        axes[0].set_title(f'CPA vs. DE similarity (Pearson r = {r:.3f})')
        axes[0].legend(fontsize=8)

        # Heatmap of shared top 40 perturbations in CPA similarity space
        top_perts_cpa = adata_pe.obs['n_cells'].fillna(0).nlargest(40).index.tolist()
        top_perts_cpa = [p for p in top_perts_cpa if p in cpa_sim_df.index]
        top_perts_cpa = top_perts_cpa[:40]

        sim_sub = cpa_sim_df.loc[top_perts_cpa, top_perts_cpa]
        sns.heatmap(
            sim_sub, cmap='RdBu_r', center=0, vmin=-0.5, vmax=1.0,
            ax=axes[1], xticklabels=True, yticklabels=True, linewidths=0.2,
        )
        axes[1].set_title(f'CPA embedding similarity\n(top {len(top_perts_cpa)} perturbations by cell count)')
        axes[1].tick_params(axis='both', labelsize=7)

        plt.suptitle('CPA vs. DE-based perturbation similarity', y=1.02)
        plt.tight_layout()
        plt.savefig(FIG_DIR / '01e_cpa_vs_de_similarity.pdf', bbox_inches='tight')
        plt.show()
else:
    print(f'DE similarity matrix not found at {de_sim_path}.')
    print('Run 01c_differential_expression.ipynb first to generate it.')
    print('Skipping CPA vs. DE comparison — plotting CPA similarity only.')

    # Fallback: just show CPA similarity for top perturbations
    top_perts_cpa = adata_pe.obs['n_cells'].fillna(0).nlargest(40).index.tolist()
    top_perts_cpa = [p for p in top_perts_cpa if p in cpa_sim_df.index][:40]
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(
        cpa_sim_df.loc[top_perts_cpa, top_perts_cpa],
        cmap='RdBu_r', center=0, vmin=-0.5, vmax=1.0,
        ax=ax, xticklabels=True, yticklabels=True, linewidths=0.2,
    )
    ax.set_title('CPA embedding cosine similarity (top 40 perturbations)')
    ax.tick_params(axis='both', labelsize=7)
    plt.tight_layout()
    plt.savefig(FIG_DIR / '01e_cpa_similarity_heatmap.pdf', bbox_inches='tight')
    plt.show()

<a id='cpa-predict'></a>
## 6. CPA: In Silico Perturbation Prediction

We validate the CPA model by predicting the gene expression of **held-out perturbations**
and comparing the predictions to the observed profiles.

### Methodology

1. Select a subset of perturbations for prediction (those with many cells for reliable
   "ground truth" from the observed data)
2. For each target perturbation:
   a. Encode a set of **control cells** to get their basal states `z_basal`
   b. Add the target perturbation embedding `z_pert` to each basal state
   c. Decode to get predicted perturbed expression
   d. Compare predicted to observed perturbed expression

### What good predictions look like

- **R² > 0.7** on the normalised expression of the top 100 differentially expressed genes
  is generally considered a good CPA prediction (Lotfollahi et al. 2023)
- **Pearson correlation > 0.8** on mean expression profiles across all HVGs
- Strongly up-regulated genes should be predicted as up, strongly down-regulated as down

### What predictions cannot capture

- Perturbations in the tail of the distribution (very strong or very weak effects)
  may be harder to predict accurately
- Perturbations of genes that are part of complex feedback circuits may show non-additive
  effects that the additive CPA model cannot capture

In [ ]:
# ---- Select perturbations for held-out prediction -------------------------
# Pick the 5 perturbations with the most cells (best ground truth statistics)
# that are NOT the control.
pert_cell_counts = adata.obs['perturbation'].value_counts()
top_perts_predict = [
    p for p in pert_cell_counts.index
    if p != 'control' and p in pert_embed_df.index
][:5]
print(f'Perturbations selected for in silico prediction:')
for p in top_perts_predict:
    print(f'  {p:20s}  ({pert_cell_counts[p]:,} cells)')

# ---- Generate predictions via get_normalized_expression -------------------
# CPA's get_normalized_expression allows specifying a target perturbation.
# We pass control cells and ask the model to predict expression under each target.
#
# The returned expression is log-normalised (same scale as adata.X),
# allowing direct comparison with observed expression.

ctrl_cells = adata_cpa[adata_cpa.obs['perturbation'] == 'control'].copy()
print(f'\nUsing {len(ctrl_cells)} control cells as the source population.')

prediction_results = []

for target_pert in top_perts_predict:
    try:
        # Predict: take control cells' basal states, apply target perturbation embedding
        # The model returns a (n_ctrl_cells, n_genes) normalised expression matrix
        predicted = model.get_normalized_expression(
            ctrl_cells,
            transform_batch=target_pert,   # switch perturbation label to the target
            library_size='latent',         # use the model's internal library size estimate
        )
        # predicted is a DataFrame: cells × genes (log1p-normalised scale)
        pred_mean = predicted.mean(axis=0)   # mean predicted expression across cells

        # Observed: mean expression of cells with the actual perturbation
        obs_cells = adata[adata.obs['perturbation'] == target_pert]
        # Use log-normalised X from main adata; subset to HVG genes in predicted
        obs_genes = [g for g in predicted.columns if g in adata.var_names]
        obs_X = adata[obs_cells.obs_names, obs_genes].X
        if sp.issparse(obs_X):
            obs_X = obs_X.toarray()
        obs_mean = obs_X.mean(axis=0)

        # Correlation between predicted and observed mean profiles
        pred_vec = pred_mean[obs_genes].values if hasattr(pred_mean, 'values') else np.array(pred_mean[obs_genes])
        r = np.corrcoef(pred_vec, obs_mean)[0, 1]

        prediction_results.append({
            'perturbation': target_pert,
            'n_ctrl_cells':     len(ctrl_cells),
            'n_obs_cells':      len(obs_cells),
            'pearson_r':        r,
            'pred_mean':        pred_vec,
            'obs_mean':         obs_mean,
            'genes':            obs_genes,
        })
        print(f'  {target_pert:20s}: Pearson r = {r:.3f}')

    except Exception as e:
        print(f'  {target_pert}: Prediction failed — {e}')

# ---- Scatter plots: predicted vs. observed mean expression ----------------
if prediction_results:
    n_preds = len(prediction_results)
    fig, axes = plt.subplots(1, n_preds, figsize=(5 * n_preds, 4.5))
    if n_preds == 1:
        axes = [axes]

    for ax, res in zip(axes, prediction_results):
        ax.scatter(
            res['obs_mean'], res['pred_mean'],
            s=4, alpha=0.4, color='steelblue', rasterized=True
        )
        # y = x reference line
        lim = [min(res['obs_mean'].min(), res['pred_mean'].min()),
               max(res['obs_mean'].max(), res['pred_mean'].max())]
        ax.plot(lim, lim, 'r--', lw=0.8)
        ax.set_xlabel('Observed mean expression (log-normalised)')
        ax.set_ylabel('CPA predicted mean expression')
        ax.set_title(f'{res["perturbation"]}\nPearson r = {res["pearson_r"]:.3f}\n'
                     f'(n_obs={res["n_obs_cells"]}, n_ctrl={res["n_ctrl_cells"]})',
                     fontsize=9)

    plt.suptitle('CPA in silico prediction vs. observed\n(each dot = one HVG)',
                 y=1.02, fontsize=12)
    plt.tight_layout()
    plt.savefig(FIG_DIR / '01e_cpa_predictions.pdf', bbox_inches='tight')
    plt.show()

    pred_summary = pd.DataFrame([
        {'perturbation': r['perturbation'],
         'pearson_r': r['pearson_r'],
         'n_obs_cells': r['n_obs_cells']}
        for r in prediction_results
    ])
    print('\nPrediction summary:')
    print(pred_summary.to_string(index=False))

<a id='cinemaot-background'></a>
## 7. Background: CINEMA-OT

### 8a. The fundamental challenge: confounding in perturbation data

In a Perturb-seq experiment, cells differ along multiple axes simultaneously:

1. **The perturbation effect** — what we want to measure
2. **Cell cycle phase** — a strong source of gene expression variation (hundreds of
   cell-cycle genes vary systematically through G1/S/G2/M)
3. **Basal expression heterogeneity** — stochastic differences in housekeeping gene
   expression unrelated to perturbation
4. **Technical noise** — dropout, sequencing depth variation

When we compare perturbed cells to control cells, we want to isolate axis 1.
But perturbed and control cells also differ on axes 2–4, which can confound our
inference if they are unequally distributed between groups.

**Example of confounding:**
Suppose a CRISPRi knockdown of a cell-cycle gene causes cells to arrest in S phase.
When we compare these arrested cells to a mixed-phase control population, we observe
hundreds of differentially expressed genes. But are these expression changes *caused*
by the knockdown directly, or are they *caused* by the secondary effect (S-phase arrest)?
Standard DE analysis cannot distinguish these possibilities.

CINEMA-OT addresses this by first **adjusting for confounders** (like cell cycle)
and then computing the causal perturbation effect on each gene.

---

### 8b. Optimal transport for matching cells

**Optimal transport (OT)** is a mathematical framework for finding the most efficient
"transport plan" that maps one probability distribution to another.

In the context of single-cell data:
- **Source distribution:** Perturbed cells in some embedding space (e.g., first few PCs
  of cell cycle genes — the "confounder" space)
- **Target distribution:** Control cells in the same confounder space
- **Transport plan:** A coupling matrix $\gamma \in \mathbb{R}^{n_{\text{pert}} \times n_{\text{ctrl}}}$
  where $\gamma_{ij}$ is the "matching weight" between perturbed cell $i$ and control cell $j$

The OT plan finds the coupling that:
1. Minimises the total transport cost (Wasserstein distance)
2. Respects marginal constraints (each cell is "matched" to its fair share of cells)

**Why this helps with confounding:**
By matching in the **confounder space** (e.g., cell cycle), OT pairs each perturbed
cell with control cells at the same cell cycle phase. The perturbation effect is then
estimated as the difference in the **residual expression** (the expression variance
NOT explained by the confounder).

This is conceptually similar to a **stratified analysis** in epidemiology: instead
of comparing all cases to all controls, you compare cases to controls matched on the
potential confounder.

---

### 8c. Causal gene programs from the transport plan

Once the OT plan $\gamma$ is computed, CINEMA-OT extracts **causal gene programs**
using non-negative matrix factorisation (NMF) on the transport-corrected expression:

1. **Compute counterfactual expression:** For each perturbed cell $i$, compute what
   its expression would have looked like without the perturbation, by taking the
   OT-weighted average of its matched control cells

2. **Compute perturbation effect per cell:** Difference = observed - counterfactual.
   This is the cell-specific, confounder-adjusted perturbation effect.

3. **Factorise with NMF:** Decompose the (cells × genes) effect matrix into
   `n_programs` factors:
   - Each **factor** is a gene program — a set of genes that co-vary in response
     to the perturbation
   - Each **cell** has a score for each program (how strongly that program is
     activated in that cell)

**Key advantage over CPA programs:**
- CINEMA-OT programs are derived from individual cell heterogeneity in perturbation
  response — some cells may activate program 1 more strongly, others program 2
- CPA assigns a single embedding to all cells with the same perturbation label,
  capturing the average effect
- CINEMA-OT can reveal **subpopulation-specific** responses within a perturbation group

The `pertpy` implementation:

```
pertpy documentation reference:
pertpy.readthedocs.io → Tools → CINEMA-OT
```

> Dong, M. et al. (2023). Causal identification of single-cell experimental perturbation
> effects with CINEMA-OT. *Nature Methods* 20, 1769–1779.

<a id='cinemaot-apply'></a>
## 8. CINEMA-OT: Application

We apply CINEMA-OT to a set of perturbations to:
1. Compute causal gene programs for each perturbation
2. Score each cell for program membership
3. Compare program structures across perturbations

### Computational considerations

CINEMA-OT solves an OT problem for each perturbation × control comparison. The cost
scales as O(n²) with the number of cells being matched. For a large screen:
- Per-perturbation OT is feasible (each group has ~200–1,000 cells + ~50K control cells)
- We subsample control cells to ≤2,000 per comparison to keep runtime manageable
- We focus on the top perturbations by number of cells (best statistics)

In [ ]:
# ---- Select perturbations for CINEMA-OT analysis -------------------------
# Choose the top perturbations by cell count for reliable OT matching.
# We limit to N_COT_PERTS for computational efficiency in this tutorial.
N_COT_PERTS = 5
cot_perts = [
    p for p in pert_cell_counts.index
    if p != 'control'
][:N_COT_PERTS]

print(f'Selected {N_COT_PERTS} perturbations for CINEMA-OT:')
for p in cot_perts:
    print(f'  {p:20s}  ({pert_cell_counts[p]:,} cells)')

# ---- Build working AnnData for CINEMA-OT ----------------------------------
# CINEMA-OT works on log-normalised expression (adata.X, not raw counts)
# and uses a low-dimensional embedding as the confounder space.
#
# Confounder representation: we use the first 20 PCA components, which
# capture shared axes of variation (cell cycle, housekeeping) that we want
# to match on, rather than the full 2,000-gene HVG space.
#
# Key CINEMA-OT arguments:
#   pert_key:     column in obs containing perturbation labels
#   control:      the control group label
#   cf_rep:       AnnData key for the confounder representation (PCA)
#   use_rep:      AnnData key for the full representation (all PCA or scVI)
#   n_programs:   number of NMF programs to extract

# Subset adata to the perturbations of interest + control
perts_and_ctrl = cot_perts + ['control']
adata_cot = adata[adata.obs['perturbation'].isin(perts_and_ctrl)].copy()

# Subsample control cells for speed (keep up to 2000 control cells)
MAX_CTRL = 2000
ctrl_idx_cot = np.where(adata_cot.obs['perturbation'] == 'control')[0]
if len(ctrl_idx_cot) > MAX_CTRL:
    rng = np.random.default_rng(42)
    ctrl_keep = rng.choice(ctrl_idx_cot, size=MAX_CTRL, replace=False)
    pert_idx_cot = np.where(adata_cot.obs['perturbation'] != 'control')[0]
    keep_idx = np.sort(np.concatenate([pert_idx_cot, ctrl_keep]))
    adata_cot = adata_cot[keep_idx].copy()
    print(f'\nSubsampled control cells to {MAX_CTRL} for OT matching.')

print(f'CINEMA-OT working AnnData: {adata_cot.n_obs:,} cells × {adata_cot.n_vars:,} genes')
print(f'  Perturbation breakdown:')
print(adata_cot.obs['perturbation'].value_counts().to_string())

# Ensure PCA is available (needed as confounder representation)
if 'X_pca' not in adata_cot.obsm:
    print('\nRecomputing PCA for CINEMA-OT subset...')
    sc.tl.pca(adata_cot, n_comps=50)

print('\nAnnData ready for CINEMA-OT.')

In [ ]:
# ---- Run CINEMA-OT -------------------------------------------------------
# CINEMA-OT API (pertpy >= 0.7.0):
#
#   cot = pt.tl.Cinemaot()
#   de = cot.causaleffect(
#       adata,
#       pert_key = 'perturbation',   # obs column with perturbation labels
#       control  = 'control',        # which label is control
#       return_matching = True,      # return the OT transport plan
#   )
#
# Returns:
#   de: AnnData of shape (n_perturbed_cells × n_genes)
#       Each row is the causal effect estimate for that cell (obs - counterfactual)
#       de.uns['matching'] contains the OT transport plan

cot_results = {}   # perturbation -> CINEMA-OT result AnnData
cot = pt.tl.Cinemaot()

for target_pert in cot_perts:
    print(f'\nRunning CINEMA-OT for {target_pert}...')

    # Subset to this perturbation + control
    adata_sub = adata_cot[
        adata_cot.obs['perturbation'].isin([target_pert, 'control'])
    ].copy()

    n_pert = (adata_sub.obs['perturbation'] == target_pert).sum()
    n_ctrl = (adata_sub.obs['perturbation'] == 'control').sum()
    print(f'  Perturbed: {n_pert}  Control: {n_ctrl}')

    try:
        # causaleffect computes:
        #   1. OT matching in confounder (PCA) space
        #   2. Cell-level causal effect estimates
        #   3. NMF factorisation into n_programs gene programs
        de_cot = cot.causaleffect(
            adata_sub,
            pert_key='perturbation',
            control='control',
            return_matching=True,
            cf_rep='X_pca',          # confounder representation: PCA
            use_rep='X_pca',         # full representation for causal effect estimation
            n_programs=CINEMAOT_N_PROGRAMS,
        )
        cot_results[target_pert] = de_cot
        print(f'  Done. Result shape: {de_cot.shape}')

    except Exception as e:
        print(f'  CINEMA-OT failed for {target_pert}: {e}')
        print('  Possible causes:')
        print('    - Too few cells for reliable OT matching')
        print('    - NMF did not converge (try n_programs=5)')
        print('    - pertpy API version mismatch (check pertpy.readthedocs.io)')

print(f'\nCINEMA-OT completed for {len(cot_results)}/{len(cot_perts)} perturbations.')

In [ ]:
# ---- Extract and visualise gene programs from CINEMA-OT ------------------
#
# For each successful CINEMA-OT run, we extract:
#   1. The NMF gene loadings (genes × programs) — which genes define each program
#   2. The NMF cell scores (cells × programs) — how strongly each cell activates each program
#   3. The top genes for each program (the "program signature")
#
# These are stored in de_cot.varm['NMF_W'] (gene loadings) and
# de_cot.obsm['NMF_H'] (cell scores), following the standard AnnData convention.

program_summaries = {}  # perturbation -> DataFrame of top genes per program

for pert, de_cot in cot_results.items():
    print(f'\nPrograms for {pert}:')

    # Locate NMF outputs (key names may vary by pertpy version)
    gene_loading_key = next(
        (k for k in de_cot.varm.keys() if 'nmf' in k.lower() or 'program' in k.lower()), None
    )
    cell_score_key = next(
        (k for k in de_cot.obsm.keys() if 'nmf' in k.lower() or 'program' in k.lower()), None
    )

    if gene_loading_key is None:
        # Fallback: check in uns
        gene_loading_key = next(
            (k for k in de_cot.uns.keys() if 'nmf' in k.lower() or 'W' in k), None
        )

    if gene_loading_key is None:
        print(f'  Could not find NMF gene loadings in de_cot.varm or de_cot.uns.')
        print(f'  Available varm keys: {list(de_cot.varm.keys())}')
        print(f'  Available uns keys: {list(de_cot.uns.keys())}')
        print('  Falling back to using raw causal effect matrix as program proxy.')

        # If NMF outputs are not in the expected place, use the mean causal effect
        # across all cells as a single "program"
        mean_effect = np.array(de_cot.X.mean(axis=0)).flatten()
        top_up   = de_cot.var_names[np.argsort(mean_effect)[-10:][::-1]].tolist()
        top_down = de_cot.var_names[np.argsort(mean_effect)[:10]].tolist()
        print(f'  Top up-regulated genes (mean causal effect):   {top_up}')
        print(f'  Top down-regulated genes (mean causal effect): {top_down}')
        program_summaries[pert] = pd.DataFrame(
            {'gene': top_up, 'direction': 'up', 'perturbation': pert}
        )
        continue

    # Extract gene loadings matrix (genes × n_programs)
    W = (
        de_cot.varm[gene_loading_key]
        if gene_loading_key in de_cot.varm
        else de_cot.uns[gene_loading_key]
    )
    if hasattr(W, 'toarray'):
        W = W.toarray()
    W = np.array(W)

    n_prog_actual = W.shape[1] if W.ndim > 1 else 1
    print(f'  Gene loading matrix (W): {W.shape}  ({W.shape[0]} genes × {n_prog_actual} programs)')

    # For each program, find the top 10 genes (highest loading = most strongly associated)
    prog_rows = []
    for prog_idx in range(n_prog_actual):
        loadings = W[:, prog_idx] if W.ndim > 1 else W
        top_gene_idx = np.argsort(loadings)[-10:][::-1]
        top_gene_names = de_cot.var_names[top_gene_idx].tolist()
        top_gene_scores = loadings[top_gene_idx].tolist()

        print(f'  Program {prog_idx}: {top_gene_names[:5]}  (top 5 of 10)')
        for g, s in zip(top_gene_names, top_gene_scores):
            prog_rows.append({
                'perturbation': pert, 'program': prog_idx,
                'gene': g, 'loading': s,
            })

    program_summaries[pert] = pd.DataFrame(prog_rows)

# Concatenate all program summaries
if program_summaries:
    all_programs = pd.concat(program_summaries.values(), ignore_index=True)
    print(f'\nTotal gene-program entries extracted: {len(all_programs)}')
else:
    all_programs = pd.DataFrame()
    print('\nNo program summaries to concatenate.')

In [ ]:
# ---- Visualise CINEMA-OT results ----------------------------------------
#
# For each perturbation with successful CINEMA-OT results, we make two plots:
#
#   1. Heatmap: cells × programs (cell score matrix H)
#      - Shows which cells have high activation of each program
#      - If cells cluster by program activation, programs are well-defined
#
#   2. UMAP of causal effect space
#      - Run UMAP on the causal effect matrix (cells × genes)
#      - Colour by program activation score

for pert, de_cot in cot_results.items():
    if pert not in program_summaries or program_summaries[pert].empty:
        continue

    # Locate cell score matrix H (cells × programs)
    cell_score_key = next(
        (k for k in de_cot.obsm.keys() if 'nmf' in k.lower() or 'program' in k.lower()), None
    )
    if cell_score_key is None:
        print(f'{pert}: No cell score matrix found in obsm — skipping heatmap.')
        continue

    H = np.array(de_cot.obsm[cell_score_key])  # (cells × programs)
    if H.ndim == 1:
        H = H.reshape(-1, 1)

    n_prog_viz = min(H.shape[1], 5)  # show at most 5 programs

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: heatmap of cell scores (subsample cells for visibility)
    N_CELLS_HM = min(200, H.shape[0])
    rng = np.random.default_rng(42)
    cell_idx = rng.choice(H.shape[0], size=N_CELLS_HM, replace=False)
    sns.heatmap(
        H[cell_idx, :n_prog_viz].T,
        cmap='viridis', ax=axes[0],
        xticklabels=False,
        yticklabels=[f'Program {i}' for i in range(n_prog_viz)],
    )
    axes[0].set_xlabel(f'Cells (n={N_CELLS_HM} sampled)')
    axes[0].set_title(f'{pert}: cell × program scores\n(rows=programs, cols=cells)')

    # Right: program activation bar chart (mean score per program)
    mean_scores = H[:, :n_prog_viz].mean(axis=0)
    axes[1].bar(range(n_prog_viz), mean_scores, color='steelblue')
    axes[1].set_xticks(range(n_prog_viz))
    axes[1].set_xticklabels([f'Program {i}' for i in range(n_prog_viz)])
    axes[1].set_ylabel('Mean activation score across cells')
    axes[1].set_title(f'{pert}: mean program activation')

    # Overlay top genes text for the strongest program
    top_prog = int(np.argmax(mean_scores))
    top_genes_prog = (
        program_summaries[pert]
        .query(f'program == {top_prog}')
        .nlargest(5, 'loading')['gene']
        .tolist()
    )
    if top_genes_prog:
        axes[1].text(
            top_prog, mean_scores[top_prog] * 0.95,
            '\n'.join(top_genes_prog),
            ha='center', va='top', fontsize=7, color='white', fontweight='bold',
        )

    plt.suptitle(f'CINEMA-OT gene programs — {pert}', y=1.02)
    plt.tight_layout()
    plt.savefig(FIG_DIR / f'01e_cinemaot_programs_{pert}.pdf', bbox_inches='tight')
    plt.show()

<a id='comparison'></a>
## 9. Program Comparison: CPA vs. CINEMA-OT

CPA and CINEMA-OT answer the same question — "what gene programs are activated by this
perturbation?" — but using very different approaches:

| Property | CPA | CINEMA-OT |
|----------|-----|----------|
| **Learning paradigm** | Supervised VAE, trained on all perturbations jointly | Per-perturbation OT matching, no model training |
| **Resolution** | One embedding vector per perturbation (average effect) | Cell-level causal effect estimate |
| **Confounder handling** | Adversarial disentanglement (removes cell-cycle etc. from basal) | Explicit OT matching in confounder space |
| **Gene programs** | Derived from the decoder (which genes change most when embedding shifts) | Directly from NMF of causal effect matrix |
| **Scalability** | Scales to any number of perturbations (embedding per perturbation) | Scales to number of cells (OT matrix is n × n) |
| **When to trust CPA more** | Many similar perturbations exist (embedding borrows strength across perturbations) | |
| **When to trust CINEMA-OT more** | You have heterogeneous perturbation responses (subpopulation effects) | |

### Comparing gene program membership

A direct comparison: for each perturbation, are the top genes from the CPA-derived
effect signature the same as the top genes from CINEMA-OT programs?

We derive the CPA gene signature by asking: "for this perturbation's embedding, which
genes change most in the decoded expression?" This is done by predicting expression
under the target perturbation vs. control and computing the difference.

In [ ]:
# ---- Derive CPA gene signatures from in silico predictions ----------------
#
# For each perturbation, the CPA gene signature is the predicted log fold-change
# between perturbed and control conditions.
# This is directly comparable to CINEMA-OT's causal effect estimate.

# Overlap perturbations present in both analyses
shared_perts = [p for p in cot_perts if p in cot_results and p in pert_embed_df.index]
print(f'Perturbations with both CPA and CINEMA-OT results: {shared_perts}')

comparison_rows = []

for pert in shared_perts:
    print(f'\nComparing programs for {pert}:')

    # ---- CPA top genes: from in silico prediction -------------------------
    if prediction_results:
        pred_res = next((r for r in prediction_results if r['perturbation'] == pert), None)
        if pred_res is not None:
            lfc_cpa = pred_res['pred_mean'] - np.array([
                adata[
                    adata.obs['perturbation'] == 'control',
                    pred_res['genes']
                ].X.toarray() if sp.issparse(
                    adata[adata.obs['perturbation'] == 'control', pred_res['genes']].X
                ) else adata[adata.obs['perturbation'] == 'control', pred_res['genes']].X
            ]).mean(axis=0)

            n_top = 20
            top_cpa_up   = np.array(pred_res['genes'])[np.argsort(lfc_cpa)[-n_top:][::-1]].tolist()
            top_cpa_down = np.array(pred_res['genes'])[np.argsort(lfc_cpa)[:n_top]].tolist()
            print(f'  CPA top {n_top} up:   {top_cpa_up[:5]}')
            print(f'  CPA top {n_top} down: {top_cpa_down[:5]}')
        else:
            print(f'  No CPA prediction for {pert} — skipping comparison.')
            continue
    else:
        print('  No CPA prediction results available — run Section 6 first.')
        continue

    # ---- CINEMA-OT top genes: from gene programs --------------------------
    if pert in program_summaries and not program_summaries[pert].empty:
        ps = program_summaries[pert]
        # If programs are labelled, take the top program by total loading
        if 'program' in ps.columns:
            top_prog = ps.groupby('program')['loading'].sum().idxmax()
            cot_top_genes = ps[ps['program'] == top_prog].nlargest(n_top, 'loading')['gene'].tolist()
        else:
            cot_top_genes = ps.nlargest(n_top, 'loading')['gene'].tolist()
        print(f'  CINEMA-OT top genes: {cot_top_genes[:5]}')
    else:
        print(f'  No CINEMA-OT program for {pert} — skipping comparison.')
        continue

    # ---- Overlap analysis ------------------------------------------------
    cpa_set = set(top_cpa_up)  # Using up-regulated as comparison
    cot_set = set(cot_top_genes)
    overlap  = cpa_set & cot_set
    jaccard  = len(overlap) / len(cpa_set | cot_set) if (cpa_set | cot_set) else 0

    comparison_rows.append({
        'perturbation':        pert,
        'n_top_cpa':           len(cpa_set),
        'n_top_cot':           len(cot_set),
        'n_overlap':           len(overlap),
        'jaccard':             jaccard,
        'overlap_genes':       sorted(overlap),
        'cpa_only':            sorted(cpa_set - cot_set),
        'cot_only':            sorted(cot_set - cpa_set),
    })
    print(f'  Overlap: {len(overlap)}/{n_top} genes  (Jaccard = {jaccard:.3f})')
    if overlap:
        print(f'  Shared genes: {sorted(overlap)}')

# Compile comparison table
if comparison_rows:
    comparison_df = pd.DataFrame(comparison_rows)
    print('\nCPA vs. CINEMA-OT gene program overlap:')
    print(comparison_df[['perturbation', 'n_top_cpa', 'n_top_cot', 'n_overlap', 'jaccard']]
          .to_string(index=False))

    # Save the comparison
    comparison_df.drop(columns=['overlap_genes', 'cpa_only', 'cot_only'])\
                 .to_csv(TBL_DIR / '01e_cpa_vs_cinemaot_comparison.csv', index=False)
    print(f'\nComparison table saved to {TBL_DIR}/01e_cpa_vs_cinemaot_comparison.csv')
else:
    comparison_df = pd.DataFrame()
    print('No valid comparisons to summarise.')

In [ ]:
# ---- Visualise CPA vs. CINEMA-OT overlap summary --------------------------
if not comparison_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Left: Jaccard similarity per perturbation
    axes[0].barh(
        range(len(comparison_df)),
        comparison_df['jaccard'],
        color='steelblue', edgecolor='none',
    )
    axes[0].set_yticks(range(len(comparison_df)))
    axes[0].set_yticklabels(comparison_df['perturbation'].tolist(), fontsize=9)
    axes[0].set_xlabel('Jaccard similarity (top gene overlap)')
    axes[0].set_title('CPA vs. CINEMA-OT gene overlap\n(higher = more agreement)')
    axes[0].axvline(0.5, color='red', ls='--', lw=1, label='Jaccard = 0.5')
    axes[0].legend(fontsize=8)

    # Right: Stacked bar — CPA only / shared / CINEMA-OT only
    x_pos = np.arange(len(comparison_df))
    n_cpa_only = comparison_df['n_top_cpa'] - comparison_df['n_overlap']
    n_shared   = comparison_df['n_overlap']
    n_cot_only = comparison_df['n_top_cot'] - comparison_df['n_overlap']

    axes[1].bar(x_pos, n_cpa_only, label='CPA only',  color='steelblue', alpha=0.8)
    axes[1].bar(x_pos, n_shared,   bottom=n_cpa_only, label='Shared',    color='purple', alpha=0.8)
    axes[1].bar(x_pos, n_cot_only, bottom=n_cpa_only + n_shared,
                label='CINEMA-OT only', color='orange', alpha=0.8)
    axes[1].set_xticks(x_pos)
    axes[1].set_xticklabels(comparison_df['perturbation'].tolist(), rotation=30, ha='right', fontsize=9)
    axes[1].set_ylabel('Number of top genes')
    axes[1].set_title('Gene program membership\n(top genes per perturbation)')
    axes[1].legend(fontsize=8)

    plt.suptitle('CPA vs. CINEMA-OT: gene program comparison', y=1.02)
    plt.tight_layout()
    plt.savefig(FIG_DIR / '01e_cpa_cinemaot_overlap.pdf', bbox_inches='tight')
    plt.show()
else:
    print('No comparison data to plot.')

<a id='save'></a>
## 10. Save Results

In [ ]:
# ---- Save the CPA-enriched AnnData ----------------------------------------
#
# The main adata now contains:
#   adata.obsm['X_pca']           — 50-PC PCA (from 01a)
#   adata.obsm['X_scVI']          — scVI latent space (from 01b)
#   adata.obsm['X_umap_scvi']     — scVI UMAP (from 01b)
#   adata.obs['leiden_scvi']      — Leiden clusters on scVI graph (from 01b)
#
# We add the CPA Leiden cluster assignments to adata.obs before saving.

# Add CPA cluster to main adata.obs
cpa_cluster_map = adata_pe.obs['leiden_cpa'].rename('leiden_cpa')
adata.obs = adata.obs.join(
    cpa_cluster_map,
    on='perturbation',
    rsuffix='_cpa',
)
# If join produces duplicates, keep the direct column
if 'leiden_cpa' not in adata.obs.columns and 'leiden_cpa_cpa' in adata.obs.columns:
    adata.obs['leiden_cpa'] = adata.obs['leiden_cpa_cpa']

print(f'Saving main AnnData with CPA annotations to: {OUT_PATH}')
adata.write_h5ad(OUT_PATH, compression='gzip')
print(f'Saved locally ({OUT_PATH.stat().st_size / 1e9:.2f} GB).')

# ---- Save perturbation embedding matrix -----------------------------------
embed_path = TBL_DIR / '01e_cpa_perturbation_embeddings.csv.gz'
pert_embed_df.to_csv(embed_path, compression='gzip')
print(f'CPA perturbation embeddings saved to {embed_path}')

# ---- Save CPA cluster assignments -----------------------------------------
cpa_cluster_path = TBL_DIR / '01e_cpa_clusters.csv'
adata_pe.obs[['leiden_cpa']].to_csv(cpa_cluster_path)
print(f'CPA cluster assignments saved to {cpa_cluster_path}')

# ---- Save program summaries (CINEMA-OT) -----------------------------------
if not all_programs.empty:
    prog_path = TBL_DIR / '01e_cinemaot_programs.csv'
    all_programs.to_csv(prog_path, index=False)
    print(f'CINEMA-OT gene programs saved to {prog_path}')

# ---- Save CPA model weights to S3 ----------------------------------------
s3 = boto3.client('s3')

for fpath in [OUT_PATH, embed_path, cpa_cluster_path]:
    if fpath.exists():
        try:
            s3_key = f'processed/scperturb/{fpath.name}' if fpath.suffix in ['.h5ad', '.gz'] \
                     else f'results/tables/{fpath.name}'
            s3.upload_file(str(fpath), S3_BUCKET, s3_key)
        except Exception:
            pass

# Upload CPA model directory
try:
    import shutil
    model_tar = str(MODEL_DIR) + '.tar.gz'
    shutil.make_archive(str(MODEL_DIR), 'gztar', str(MODEL_DIR.parent), MODEL_DIR.name)
    s3.upload_file(model_tar, S3_BUCKET, 'results/model_checkpoints/cpa_model_rpe1.tar.gz')
    print('CPA model checkpoint uploaded to S3.')
except Exception as e:
    print(f'Model checkpoint S3 upload failed: {e}')

print('\nAll results saved.')

In [ ]:
# <a id='summary'></a>
print('=' * 70)
print('01e CPA GENE PROGRAMS — SUMMARY')
print('=' * 70)
print()

checks = [
    ('Data loaded from 01b',                             SCVI_PATH.exists() or QC_PATH.exists()),
    ('CPA AnnData prepared (raw counts, HVG, cat pert)', adata_cpa is not None),
    ('CPA model trained or loaded',                      MODEL_DIR.exists()),
    ('Perturbation embeddings extracted',                pert_embed_df is not None and len(pert_embed_df) > 0),
    ('CPA embedding UMAP computed',                      'X_umap' in adata_pe.obsm),
    ('Leiden clustering on CPA embeddings',              'leiden_cpa' in adata_pe.obs.columns),
    ('CPA vs. DE similarity comparison',                 de_sim_path.exists()),
    ('In silico perturbation predictions',               len(prediction_results) > 0),
    ('CINEMA-OT applied',                                len(cot_results) > 0),
    ('Gene programs extracted (CINEMA-OT)',              not all_programs.empty),
    ('CPA vs. CINEMA-OT comparison',                     not comparison_df.empty),
    ('Results saved locally',                            OUT_PATH.exists()),
]

all_pass = True
for check, result in checks:
    status = 'v' if result else 'X'
    print(f'  [{status}]  {check}')
    if not result:
        all_pass = False

print()
print('Key results:')
print(f'  CPA perturbation embedding: {pert_embed_df.shape} '
      f'(perturbations × {N_LATENT_DRUG} dims)')
print(f'  CPA Leiden clusters: {n_cpa_clusters}')
print(f'  In silico predictions:')
for res in prediction_results:
    print(f'    {res["perturbation"]:20s}  Pearson r = {res["pearson_r"]:.3f}')
print(f'  CINEMA-OT perturbations: {len(cot_results)} analysed')
print(f'  Gene programs per perturbation: {CINEMAOT_N_PROGRAMS}')
if not comparison_df.empty:
    print(f'  Median CPA/CINEMA-OT gene overlap (Jaccard): '
          f'{comparison_df["jaccard"].median():.3f}')

print()
print('Phase 2 CPA/CINEMA-OT checkpoint:', 'PASSED' if all_pass else 'INCOMPLETE (see above)')
print()
print('Phase 2 complete! Next steps:')
print('  → Tag the repo: git tag phase2-complete')
print('  → Begin Phase 3: notebooks/02_jin_2023_cardiac/')
print('     Apply pipeline to TianKampmann2021 and Jin 2023 cardiac datasets')

In [ ]:
from datetime import datetime
import subprocess

timestamp   = datetime.now().strftime('%Y%m%d_%H%M')
report_dir  = Path('../../results/reports')
report_dir.mkdir(parents=True, exist_ok=True)
report_path = report_dir / f'01e_cpa_gene_programs_{timestamp}.html'

nb_path = (
    __vsc_ipynb_file__
    if '__vsc_ipynb_file__' in dir()
    else '01e_cpa_gene_programs.ipynb'
)
subprocess.run(
    ['jupyter', 'nbconvert', '--to', 'html', '--output', str(report_path), nb_path],
    check=False,
)
print(f'Report saved to {report_path}')